In [11]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os
import shutil
import gymnasium as gym
import torch
import sys
sys.path.insert(0, os.path.abspath(".."))
from src.rl_2.env_adapter import MatchEnv, RandomOpponentController
from src.rl_2.model import ActorCritic
from src.rl_2.pool import PoolOpponentController
from src.rl_2.ppo import train_mappo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(1)

TEAM_SIZE = 1
NUM_ENVS = 24  # Multiprocessing across CPU cores

In [14]:
# --- Phase 1 Configurations ---
PITCH_W_P1 = 840.0
PITCH_H_P1 = 500.0
GOAL_H_P1 = 450.0  # Wide nets
ROUND_STEPS_P1 = 900  # 15s rounds
SAVE_DIR_P1 = "models/stage1/phase1"
os.makedirs(SAVE_DIR_P1, exist_ok=True)


def make_p1_env(env_rank: int):

  def _thunk():
    env = MatchEnv(
        team_size=TEAM_SIZE,
        max_round_steps=ROUND_STEPS_P1,
        goal_height=GOAL_H_P1,
        pitch_width=PITCH_W_P1,
        pitch_height=PITCH_H_P1,
        opponent_controller=RandomOpponentController(),
    )
    env.reset(seed=1000 + env_rank)
    return env

  return _thunk


envs_p1 = gym.vector.AsyncVectorEnv(
    [make_p1_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_p1 = ActorCritic().to(device)

train_mappo(
    envs=envs_p1,
    model=model_p1,
    device=device,
    team_size=TEAM_SIZE,
    total_timesteps=5_000_000,
    num_envs=NUM_ENVS,
    num_steps=340,  # Batch ~8,160
    update_epochs=2,
    minibatch_size=1024,
    lr_init=1e-3,
    lr_final=1e-5,
    ent_coef_init=0.02,
    ent_coef_final=0.002,
    gamma=0.990,
    gae_lambda=0.95,
    active_tiers=["random"],
    target_tier="random",
    filter_thresholds=None,
    goal_height=GOAL_H_P1,
    pitch_width=PITCH_W_P1,
    pitch_height=PITCH_H_P1,
    save_dir=SAVE_DIR_P1,
    pool_dir=None,
    eval_episodes=40,
    eval_freq=200_000,
    max_steps=ROUND_STEPS_P1,
)

envs_p1.close()

🚀 MAPPO Initialized | Format: 1v1 | Envs: 24 | Step Batch: 8160 | Device: cuda

📊 [EVALUATION @ Step 204,000 | SPS: 4330 | Tiers: ['random']]
   ⚔️  vs Random    [TARGET] | WR:  17.5% | Reward: +0.250 | Goals: 14 Scored, 2 Conceded (+12 Net)
🏆 NEW CHAMPION REGISTERED @ step 204,000 -> models/stage1/phase1/pool/history_204000.pt
   ⭐⭐ PROMOTED! New Best Score (random) -> [WR: 17.5%, Reward: +0.250, Net: +12]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage1/phase1/best_model.pt

📊 [EVALUATION @ Step 408,000 | SPS: 3884 | Tiers: ['random']]
   ⚔️  vs Random    [TARGET] | WR:  40.0% | Reward: +0.588 | Goals: 38 Scored, 14 Conceded (+24 Net)
🏆 NEW CHAMPION REGISTERED @ step 408,000 -> models/stage1/phase1/pool/history_408000.pt
   ⭐⭐ PROMOTED! New Best Score (random) -> [WR: 40.0%, Reward: +0.588, Net: +24]
      (Defeated previous record: [WR: 17.5%, Reward: +0.250]) -> Saved: models/stage1/phase1/best_model.pt

📊 [EVALUATION @ Step 603,840 | SPS: 3797 | T

In [16]:
# --- Phase 2 Configurations ---
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0  # Regulation net
ROUND_STEPS_P2 = 1800 # 30s continuous match
SAVE_DIR_P2 = "models/stage1/phase2"
os.makedirs(SAVE_DIR_P2, exist_ok=True)


def make_p2_env(env_rank: int):

  def _thunk():
    env = MatchEnv(
        team_size=TEAM_SIZE,
        max_round_steps=ROUND_STEPS_P2,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=RandomOpponentController(),
    )
    env.reset(seed=2000 + env_rank)
    return env

  return _thunk


envs_p2 = gym.vector.AsyncVectorEnv(
    [make_p2_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

# Warmstart from Phase 1 Champion
model_p2 = ActorCritic().to(device)
phase1_best = os.path.join(SAVE_DIR_P1, "best_model.pt")

if os.path.exists(phase1_best):
  ckpt = torch.load(phase1_best, map_location=device, weights_only=False)
  state_dict = (
      ckpt["model_state_dict"]
      if isinstance(ckpt, dict) and "model_state_dict" in ckpt
      else ckpt
  )
  model_p2.load_state_dict(state_dict, strict=False)
  print(f"🔥 Warmstarted Phase 2 from: {phase1_best}")

train_mappo(
    envs=envs_p2,
    model=model_p2,
    device=device,
    team_size=TEAM_SIZE,
    total_timesteps=10_000_000,
    num_envs=NUM_ENVS,
    num_steps=512,  # Extended buffer for 30s matches
    update_epochs=2,
    minibatch_size=1024,
    lr_init=1.5e-4,  # Lower starting LR preserves striking skills
    lr_final=1e-5,
    ent_coef_init=0.010,
    ent_coef_final=0.002,
    gamma=0.996,  # Scaled for 450 decision steps
    gae_lambda=0.97,
    active_tiers=["random"],
    target_tier="random",
    filter_thresholds=None,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_P2,
    pool_dir=None,
    eval_episodes=40,
    eval_freq=200_000,
    max_steps=ROUND_STEPS_P2,
)

envs_p2.close()

🔥 Warmstarted Phase 2 from: models/stage1/phase1/best_model.pt
🚀 MAPPO Initialized | Format: 1v1 | Envs: 24 | Step Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 208,896 | SPS: 4531 | Tiers: ['random']]
   ⚔️  vs Random    [TARGET] | WR:  32.5% | Reward: +0.677 | Goals: 28 Scored, 0 Conceded (+28 Net)
🏆 NEW CHAMPION REGISTERED @ step 208,896 -> models/stage1/phase2/pool/history_208896.pt
   ⭐⭐ PROMOTED! New Best Score (random) -> [WR: 32.5%, Reward: +0.677, Net: +28]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage1/phase2/best_model.pt

📊 [EVALUATION @ Step 405,504 | SPS: 3834 | Tiers: ['random']]
   ⚔️  vs Random    [TARGET] | WR:  35.0% | Reward: +0.705 | Goals: 28 Scored, 0 Conceded (+28 Net)
🏆 NEW CHAMPION REGISTERED @ step 405,504 -> models/stage1/phase2/pool/history_405504.pt
   ⭐⭐ PROMOTED! New Best Score (random) -> [WR: 35.0%, Reward: +0.705, Net: +28]
      (Defeated previous record: [WR: 32.5%, Reward: +0.677]) -> Saved: models/stage1/pha

In [ ]:
# --- Phase 3 Configurations ---
SAVE_DIR_P3 = "models/stage1/phase3"
POOL_DIR_P3 = os.path.join(SAVE_DIR_P3, "pool")
os.makedirs(POOL_DIR_P3, exist_ok=True)

# 1. Seed Opponent Pool with Phase 2 Champion
phase2_best = os.path.join(SAVE_DIR_P2, "best_model.pt")
if os.path.exists(phase2_best):
  shutil.copy(phase2_best, os.path.join(POOL_DIR_P3, "champion.pt"))
  shutil.copy(phase2_best, os.path.join(POOL_DIR_P3, "history_0.pt"))
  print(f"✅ Initialized Phase 3 opponent pool using Phase 2 weights.")


# 2. Environments with Opponent Sampling
def make_p3_env(env_rank: int):

  def _thunk():
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_P3,
        team="blue",
        device="cpu",
        p_random=0.05,
        p_heuristic=0.90,  # Heavy sparring against deterministic angles
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        max_round_steps=ROUND_STEPS_P2,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
    )
    env.reset(seed=3000 + env_rank)
    return env

  return _thunk


envs_p3 = gym.vector.AsyncVectorEnv(
    [make_p3_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

# 3. Warmstart Learner Model
model_p3 = ActorCritic().to(device)
if os.path.exists(phase2_best):
  ckpt = torch.load(phase2_best, map_location=device, weights_only=False)
  state_dict = (
      ckpt["model_state_dict"]
      if isinstance(ckpt, dict) and "model_state_dict" in ckpt
      else ckpt
  )
  model_p3.load_state_dict(state_dict, strict=False)
  print(f"🔥 Warmstarted Learner Model from Phase 2: {phase2_best}")

# 4. Train Phase 3 (Gatekeeper Gauntlet & Dethroning)
train_mappo(
    envs=envs_p3,
    model=model_p3,
    device=device,
    team_size=TEAM_SIZE,
    total_timesteps=25_000_000,
    num_envs=NUM_ENVS,
    num_steps=512,
    update_epochs=2,
    minibatch_size=1024,
    lr_init=1.5e-4,
    lr_final=1e-5,
    ent_coef_init=0.010,
    ent_coef_final=0.002,
    gamma=0.996,
    gae_lambda=0.97,
    active_tiers=["heuristic"],
    target_tier="heuristic",
    filter_thresholds={
        "heuristic": 0.80,  # Qualification gate
    },
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_P3,
    pool_dir=POOL_DIR_P3,
    eval_episodes=40,
    eval_freq=200_000,
    max_steps=ROUND_STEPS_P2,
)

envs_p3.close()

✅ Initialized Phase 3 opponent pool using Phase 2 weights.
🔥 Warmstarted Learner Model from Phase 2: models/stage1/phase2/best_model.pt
🚀 MAPPO Initialized | Format: 1v1 | Envs: 24 | Step Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 208,896 | SPS: 3582 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  75.0% | Reward: +1.655 | Goals: 86 Scored, 18 Conceded (+68 Net)
   ❌ Retaining current baseline. Did not pass criteria for heuristic: [WR: None, Reward: None, Net: None]

📊 [EVALUATION @ Step 405,504 | SPS: 3079 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  82.5% | Reward: +1.873 | Goals: 84 Scored, 10 Conceded (+74 Net)
🏆 NEW CHAMPION REGISTERED @ step 405,504 -> models/stage1/phase3/pool/history_405504.pt
   ⭐⭐ PROMOTED! New Best Score (heuristic) -> [WR: 82.5%, Reward: +1.873, Net: +74]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage1/phase3/best_model.pt

📊 [EVALUATION @ Step 602,112 | SPS: 2879 | Tiers: ['heuristic']]
  